# 03 — RouteLLM (strong / weak via LiteLLM)

[RouteLLM](https://github.com/lm-sys/RouteLLM) is a pretrained **strong vs weak** cost/quality classifier. LiteLLM is how it talks to providers.

This is **not** greeting-vs-complex. Misalignment with our labels is a valid finding.

RouteLLM is one pair per `Controller`. This notebook runs two pairs so we can see a "set" of weaks:

- pair A: OpenAI strong vs `OLLAMA_MODEL` weak
- pair B: OpenAI strong vs `OLLAMA_MODEL_2` weak

First `mf` load downloads `routellm/mf_gpt4_augmented` from HuggingFace.

Threshold `0.11593` is RouteLLM's published ~50% strong calibration on their eval — not tuned to us.

Kernel: Python 3.10+.


In [1]:
# %pip install routellm python-dotenv -q


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
poc_dir = None
root = None
for p in [cwd, *cwd.parents]:
    if (p / "eval_queries.py").exists():
        poc_dir = p
        break
    if (p / "poc" / "eval_queries.py").exists():
        poc_dir = p / "poc"
        break
if poc_dir is None:
    raise FileNotFoundError("eval_queries.py not found — run from route-chatbot/ or route-chatbot/poc/")
sys.path.insert(0, str(poc_dir))

for p in [cwd, *cwd.parents]:
    if (p / ".env").exists() and (p / "main.py").exists():
        root = p
        load_dotenv(p / ".env")
        break
else:
    load_dotenv()

from eval_queries import EVAL_QUERIES

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")
OLLAMA_MODEL_2 = os.getenv("OLLAMA_MODEL_2", "llama3")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")
TYPESAFE_API_KEY = os.getenv("TYPESAFE_API_KEY", "")

print("poc_dir", poc_dir)
print("eval queries", len(EVAL_QUERIES))
print("ollama", OLLAMA_BASE_URL, OLLAMA_MODEL, "| alt", OLLAMA_MODEL_2)
print("openai model", OPENAI_MODEL, "| key set", bool(OPENAI_API_KEY))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))


poc_dir /Users/tushar/PravarAI/route-chatbot/poc
eval queries 17
ollama http://localhost:11434 llama3.1:8b | alt qwen3.5:9b
openai model gpt-3.5-turbo | key set True
typesafe key set False
typesafe key set False


## Controllers


In [3]:
from routellm.controller import Controller

# RouteLLM docs use ollama_chat/<tag> for local weak models.
weak_a = f"ollama_chat/{OLLAMA_MODEL}"
weak_b = f"ollama_chat/{OLLAMA_MODEL_2}"
strong = OPENAI_MODEL
THRESHOLD = 0.11593

print("loading mf router (HuggingFace download on first run)...")
ctrl_a = Controller(routers=["mf"], strong_model=strong, weak_model=weak_a)
ctrl_b = Controller(routers=["mf"], strong_model=strong, weak_model=weak_b) if weak_b != weak_a else None
print("pair A", strong, "vs", weak_a)
print("pair B", strong, "vs", weak_b, "| skipped" if ctrl_b is None else "ready")


/Users/tushar/PravarAI/pravar-monorepo/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading mf router (HuggingFace download on first run)...
pair A gpt-3.5-turbo vs ollama_chat/llama3.1:8b
pair B gpt-3.5-turbo vs ollama_chat/qwen3.5:9b ready


## Decision-only `route()` — no generation


In [4]:
def _label(routed_model: str) -> str:
    return "openai" if routed_model == strong else "ollama"


def decide_route(message: str) -> str:
    routed = ctrl_a.route(message, router="mf", threshold=THRESHOLD)
    return _label(routed)


def decide_route_pair_b(message: str) -> str:
    if ctrl_b is None:
        return decide_route(message)
    routed = ctrl_b.route(message, router="mf", threshold=THRESHOLD)
    return _label(routed)


## Eval pair A


In [5]:
rows = []
for item in EVAL_QUERIES:
    t0 = time.perf_counter()
    err = None
    predicted = None
    extra = None
    try:
        result = decide_route(item["message"])
        if isinstance(result, tuple):
            predicted = result[0]
            extra = result[1] if len(result) > 1 else None
        else:
            predicted = result
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
    ms = (time.perf_counter() - t0) * 1000
    rows.append({
        "message": item["message"],
        "expected": item["expected"],
        "predicted": predicted,
        "match": predicted == item["expected"],
        "latency_ms": round(ms, 1),
        "error": err,
        "extra": extra,
    })

n = len(rows)
ok = sum(1 for r in rows if r["match"])
errs = sum(1 for r in rows if r["error"])
mean_ms = sum(r["latency_ms"] for r in rows) / n if n else 0
print(f"accuracy {ok}/{n} ({100 * ok / n:.0f}%)  mean latency {mean_ms:.1f} ms  errors {errs}")
print()
for r in rows:
    flag = "OK  " if r["match"] else "MISS"
    extra = f"  {r['extra']}" if r["extra"] else ""
    err = f"  ERR {r['error']}" if r["error"] else ""
    print(f"  [{flag}] {r['latency_ms']:7.1f} ms  exp={r['expected']:7} pred={r['predicted']}  {r['message'][:70]}{extra}{err}")


accuracy 14/17 (82%)  mean latency 832.2 ms  errors 0

  [OK  ]  5089.9 ms  exp=ollama  pred=ollama  hi there
  [OK  ]  3827.3 ms  exp=ollama  pred=ollama  hello
  [OK  ]   407.5 ms  exp=ollama  pred=ollama  hey
  [OK  ]  1232.2 ms  exp=ollama  pred=ollama  good morning
  [OK  ]   223.6 ms  exp=ollama  pred=ollama  thanks
  [OK  ]   288.3 ms  exp=ollama  pred=ollama  how are you
  [OK  ]   305.6 ms  exp=openai  pred=openai  write a function to reverse a linked list
  [OK  ]   307.5 ms  exp=openai  pred=openai  compare merge sort and quick sort
  [MISS]   307.5 ms  exp=openai  pred=ollama  debug this python code
  [OK  ]   208.8 ms  exp=openai  pred=openai  analyze the time complexity of this algorithm
  [MISS]   302.4 ms  exp=openai  pred=ollama  write a poem about the ocean
  [MISS]   226.0 ms  exp=openai  pred=ollama  design a strategy for caching
  [OK  ]   285.5 ms  exp=ollama  pred=ollama  tell me something interesting
  [OK  ]   308.3 ms  exp=ollama  pred=ollama  what did you do 

## Eval pair B (skipped if `OLLAMA_MODEL_2` == `OLLAMA_MODEL`)


In [6]:
if ctrl_b is None:
    print("pair B skipped — set OLLAMA_MODEL_2 to a different pulled tag to compare weaks.")
else:
    decide_route_saved = decide_route
    decide_route = decide_route_pair_b
    rows = []
    for item in EVAL_QUERIES:
        t0 = time.perf_counter()
        err = None
        predicted = None
        extra = None
        try:
            predicted = decide_route(item["message"])
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
        ms = (time.perf_counter() - t0) * 1000
        rows.append({
            "message": item["message"],
            "expected": item["expected"],
            "predicted": predicted,
            "match": predicted == item["expected"],
            "latency_ms": round(ms, 1),
            "error": err,
        })
    n = len(rows)
    ok = sum(1 for r in rows if r["match"])
    errs = sum(1 for r in rows if r["error"])
    mean_ms = sum(r["latency_ms"] for r in rows) / n if n else 0
    print(f"pair B accuracy {ok}/{n} ({100 * ok / n:.0f}%)  mean latency {mean_ms:.1f} ms  errors {errs}")
    for r in rows:
        flag = "OK  " if r["match"] else "MISS"
        err = f"  ERR {r['error']}" if r["error"] else ""
        print(f"  [{flag}] {r['latency_ms']:7.1f} ms  exp={r['expected']:7} pred={r['predicted']}  {r['message'][:70]}{err}")
    decide_route = decide_route_saved


pair B accuracy 14/17 (82%)  mean latency 245.8 ms  errors 0
  [OK  ]   286.2 ms  exp=ollama  pred=ollama  hi there
  [OK  ]   253.4 ms  exp=ollama  pred=ollama  hello
  [OK  ]   219.9 ms  exp=ollama  pred=ollama  hey
  [OK  ]   235.5 ms  exp=ollama  pred=ollama  good morning
  [OK  ]   241.0 ms  exp=ollama  pred=ollama  thanks
  [OK  ]   279.6 ms  exp=ollama  pred=ollama  how are you
  [OK  ]   307.5 ms  exp=openai  pred=openai  write a function to reverse a linked list
  [OK  ]   219.7 ms  exp=openai  pred=openai  compare merge sort and quick sort
  [MISS]   221.1 ms  exp=openai  pred=ollama  debug this python code
  [OK  ]   277.3 ms  exp=openai  pred=openai  analyze the time complexity of this algorithm
  [MISS]   229.6 ms  exp=openai  pred=ollama  write a poem about the ocean
  [MISS]   219.4 ms  exp=openai  pred=ollama  design a strategy for caching
  [OK  ]   232.8 ms  exp=ollama  pred=ollama  tell me something interesting
  [OK  ]   217.6 ms  exp=ollama  pred=ollama  what did y

## Notes (fill during the experiment)

- Strong/weak is a different question than greeting/complex. Expect misses on casual queries that still look "hard" to the mf router, and the reverse.
- Changing the weak model (pair B) can change who wins even with the same strong model.
- Router tax: local mf inference, not an extra LLM call.


In [7]:
GENERATE = True

if GENERATE:
    for item in EVAL_QUERIES[:]:
        routed = ctrl_a.route(item["message"], router="mf", threshold=THRESHOLD)
        print("---", item["message"], "->", routed)
        resp = ctrl_a.completion(
            model=f"router-mf-{THRESHOLD}",
            messages=[{"role": "user", "content": item["message"]}],
            max_tokens=64,
        )
        print((resp.choices[0].message.content or "")[:400])
        print()
else:
    print("GENERATE is False — routing only. Flip it to call Controller.completion.")


--- hi there -> ollama_chat/llama3.1:8b
It's nice to meet you. Is there something I can help you with or would you like to chat?

--- hello -> ollama_chat/llama3.1:8b
Hello! How can I assist you today?

--- hey -> ollama_chat/llama3.1:8b
What's up? Is there something on your mind that you'd like to chat about?

--- good morning -> ollama_chat/llama3.1:8b
Good morning! How are you today?

--- thanks -> ollama_chat/llama3.1:8b
You're welcome! Is there anything else I can help you with?

--- how are you -> ollama_chat/llama3.1:8b
I'm just a computer program, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions or tasks you may have! How can I assist you today?

--- write a function to reverse a linked list -> gpt-3.5-turbo
Here is a sample code in Python to reverse a linked list:

```python
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class LinkedList:
    def __init__(self